### Install requiremnts

In [3]:
!pip install -r ../requirements.txt

In [4]:
import sys
sys.path.append('../')
from checkpoints import DT, MT, LT
import os


from validate_birdset import load_model, get_test_loader, test
from checkpoints import DT, MT, LT
import numpy as np
import requests
import zipfile
import glob
import gdown

DOWN_TASKS = ["HSN"]
DEVICE = "cuda" # change to "cpu" if no gpu is available.

/home/bellafkir/Documents/sa4birds/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def run_testing(ckpts, extracted_interval=7, target_length=701, device='cuda'):
    for down_task in DOWN_TASKS:

        print(ckpts)

        models = []
        configs = []

        # ---------------------------------------------------------
        # Load models and their configs from checkpoints
        # ---------------------------------------------------------
        for ckpt in ckpts:
            m, cfg = load_model(ckpt, device=device)
            models.append(m)
            # Override validation parameters for evaluation
            cfg.frontend.val_target_length = target_length
            cfg.event_decoder.val.extracted_interval = extracted_interval
            # Set dataset name to current downstream task
            cfg.train.dataset_name = down_task
            configs.append(cfg)

        results = dict()
        # Create test dataloader using the first config
        val_loader, class_names = get_test_loader(configs[0])

        # Map class names to label IDs using the config label map
        label2id = {k: v for k, v in configs[0].train.label_map.items()}

        # Only test on classes relevant for the current dataset
        relevant = [label2id[c] for c in class_names]

        # Store metrics for each model (for later averaging)
        metrics = {"auroc": [],
                   "cmap": [],
                   "top1_acc": []}

        # ---------------------------------------------------------
        # test each model on the test dataset
        # ---------------------------------------------------------
        for model, _ in zip(models, configs):
            # Run evaluation
            auroc, cmap, top1_acc = test(model, val_loader, relevant, device=device)

            # Store metrics
            metrics["auroc"].append(auroc)
            metrics["cmap"].append(cmap)
            metrics["top1_acc"].append(top1_acc)

        # ---------------------------------------------------------
        # Aggregate metrics across all checkpoints/models
        # ---------------------------------------------------------
        results[down_task] = {key: float(np.mean(values)) for key, values in metrics.items()}
        print(results)

## 1. Impact of incorporating secondary label annotations into training, comparing model evaluation metrics with and without annotation weighting on the HSN test set (Table 5)

In [6]:
# download/load checkpoints trained without secondary labels
if not os.path.exists('../ckpts/rest/HSN_secw0'):
    gdown.download_folder('https://drive.google.com/drive/folders/1LiteKuM42wP1YuWUIAaA-bqxtTXy1GBS?usp=sharing', output="../ckpts/rest/")

# get ckpts paths 
ckpts = glob.glob('../ckpts/rest/HSN_secw0/*')

run_testing(ckpts, device=DEVICE)
    

Retrieving folder contents


Retrieving folder 1bgZwnXFbG0cjkXu40BP711dkbiVFIzZ9 HSN_eca_nfnet_l1_2026-03-09_122507
Retrieving folder 11icrGMxnjRiJ5mLLpMSBk46KhPE4iiDH models
Processing file 1xezHNho4lPXq-N9_iYs55wsDxic9Fdf1 model.pth
Retrieving folder 1BdmP8QowWsaFS1PcE9318fLb2vS2w3ce HSN_eca_nfnet_l1_2026-03-09_123657
Retrieving folder 1lXBQ5oIcZO9VxzQhcwhivWeHJ1Fg-j8u models
Processing file 1fC740wsBgjs9ImuJ7CfZntp2UM0PmCJD model.pth
Retrieving folder 16PkMXOezetDcGTLMObu_ih7IupOt7icz HSN_eca_nfnet_l1_2026-03-09_124840
Retrieving folder 14C-b09ctgzJrGSiU2V4mgGLFi9LzPIzY models
Processing file 1dzLra03XyOwX_zNX-Qm1moCWvQzmrbO- model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1xezHNho4lPXq-N9_iYs55wsDxic9Fdf1
From (redirected): https://drive.google.com/uc?id=1xezHNho4lPXq-N9_iYs55wsDxic9Fdf1&confirm=t&uuid=644bcab0-6491-4bea-8078-ce389cc788b7
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_secw0/HSN_eca_nfnet_l1_2026-03-09_122507/models/model.pth
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 230M/230M [00:02<00:00, 86.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1fC740wsBgjs9ImuJ7CfZntp2UM0PmCJD
From (redirected): https://drive.google.com/uc?id=1fC740wsBgjs9ImuJ7CfZntp2UM0PmCJD&confirm=t&uuid=21ae2140-9874-4893-bc04-d895c9be7d65
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_secw0/HSN_eca_nfnet_l1_2026-03-09_123657/models/model.pth
100%|█████████████████████████████████████████████████████████

['../ckpts/rest/HSN_secw0/HSN_eca_nfnet_l1_2026-03-09_122507', '../ckpts/rest/HSN_secw0/HSN_eca_nfnet_l1_2026-03-09_124840', '../ckpts/rest/HSN_secw0/HSN_eca_nfnet_l1_2026-03-09_123657']


2026-05-05 10:32:16,935 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:32:17,103 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:32:17,116 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:32:17,504 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:32:17,622 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:32:17,625 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:32:18,048 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:32:18,167 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9502559753896435, 'cmap': 0.5623522065687582, 'top1_acc': 0.7183063626289368}}


In [7]:
if not os.path.exists('../ckpts/DT/HSN'):
        gdown.download_folder('https://drive.google.com/drive/folders/1pgdC-y47iGGx6iVe5QuCC_kMiT7wkACO?usp=sharing', output="../ckpts/DT/")

# with secondary labels 
ckpts = glob.glob('../ckpts/DT/HSN/*')
run_testing(ckpts, device=DEVICE)

['../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-05-05 10:34:38,928 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:34:39,046 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:34:39,049 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:34:39,597 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:34:39,716 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:34:39,728 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:34:40,173 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:34:40,293 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9560262623085944, 'cmap': 0.5792627376801449, 'top1_acc': 0.7324406305948893}}


## 2. Performance metrics for various classifier heads tested on the HSN (Table 4)

In [8]:
# DSA 
ckpts = glob.glob('../ckpts/DT/HSN/*')
run_testing(ckpts, device=DEVICE)

['../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-05-05 10:37:04,248 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:37:04,370 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:37:04,383 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:37:04,744 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:37:04,865 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:37:04,875 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:37:05,246 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:37:05,369 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9560262623085944, 'cmap': 0.5792627376801449, 'top1_acc': 0.7324406305948893}}


In [9]:
# Linear

if not os.path.exists('../ckpts/rest/HSN_linearhead'):
    gdown.download_folder('https://drive.google.com/drive/folders/1zun4_eNfs_l0aXKWPR-ujmTy7LHsc5k5?usp=sharing', output="../ckpts/rest/")


ckpts = glob.glob('../ckpts/rest/HSN_linearhead/*')
run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 1e5th_6EpNvcXXfuvN1862hep1ad5TG3S HSN_eca_nfnet_l1_2026-03-09_150746
Retrieving folder 1mJmx64o0IfrViXVxUeufy0mRjFmayoD5 models
Processing file 1PTXty_uYkRn3J_rBB8EUbJ0a0ijOaB-M model.pth
Retrieving folder 1P5GKGXBOi2G3siAvX_40Ypvo4h9dIzf4 HSN_eca_nfnet_l1_2026-03-09_151901
Retrieving folder 1SRYMYKa2Z1wo24PryEamYrjYf1WAXv-1 models
Processing file 1wbuSzYYFqtiVUPly2iE70agqezUdCJi1 model.pth
Retrieving folder 1WO1F6H3VlsX3mOHXBP5P_NU_1E0wWAVp HSN_eca_nfnet_l1_2026-03-09_153017
Retrieving folder 1bMSOrKirNCxfaa95q_FVOcqUi00gljYE models
Processing file 1S0DnFEsguFPvez6yKMSrNsOkbRg94ncF model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1PTXty_uYkRn3J_rBB8EUbJ0a0ijOaB-M
From (redirected): https://drive.google.com/uc?id=1PTXty_uYkRn3J_rBB8EUbJ0a0ijOaB-M&confirm=t&uuid=3f4361ca-06aa-447a-a678-034ae3604bc7
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_150746/models/model.pth
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 154M/154M [00:01<00:00, 90.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1wbuSzYYFqtiVUPly2iE70agqezUdCJi1
From (redirected): https://drive.google.com/uc?id=1wbuSzYYFqtiVUPly2iE70agqezUdCJi1&confirm=t&uuid=6d14d4ed-7b2f-488a-b6da-ca53803348cf
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_151901/models/model.pth
100%|███████████████████████████████████████████████

['../ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_153017', '../ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_151901', '../ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_150746']


2026-05-05 10:39:37,149 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:39:37,268 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:39:37,281 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:39:37,575 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:39:37,706 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:39:37,719 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:39:38,011 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:39:38,134 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9120344598123685, 'cmap': 0.5234318648468855, 'top1_acc': 0.6881780425707499}}


In [10]:
run_testing(ckpts, extracted_interval=5, target_length=501, device=DEVICE)

['../ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_153017', '../ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_151901', '../ckpts/rest/HSN_linearhead/HSN_eca_nfnet_l1_2026-03-09_150746']


2026-05-05 10:41:55,866 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:41:55,983 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:41:55,999 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:41:56,281 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:41:56,401 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:41:56,415 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:41:56,699 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:41:56,822 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9261919057612878, 'cmap': 0.5521745045732435, 'top1_acc': 0.6725559631983439}}


In [11]:
# Time Attention
if not os.path.exists('../ckpts/rest/HSN_timeattention'):
    gdown.download_folder('https://drive.google.com/drive/folders/1NekF8AcJWMmKqiuYDrJoA2SewlqjmATu?usp=sharing', output="../ckpts/rest/")

ckpts = glob.glob('../ckpts/rest/HSN_timeattention/*')


run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 15SDes17j3zmTuL29Y09UAkU9SzcNy8Es HSN_eca_nfnet_l1_2026-03-09_161144
Retrieving folder 1AwAh0Gr3af9uSM3v6SXExRK588ocqDwS models
Processing file 1DJFb1AW3hrUNrXW4zm9ns__zVB8D91-A model.pth
Retrieving folder 1KOLCnGQE4o8zsIBNvX1yiPyS84gXSFDV HSN_eca_nfnet_l1_2026-03-09_162312
Retrieving folder 1FDRZUZf3GCaDSj7qmp860gWCGft_QUwI models
Processing file 1cjhVqNc4oVYkZNsgnliVnQcQ-D4pKNYF model.pth
Retrieving folder 1s3bI-awtT2WBS3QPn94iueT9YtuyxJBm HSN_eca_nfnet_l1_2026-03-09_163440
Retrieving folder 15f72cqXZPG2dFROrj_PkUPazazs605vj models
Processing file 1C_sM44ywoNUz0xIGR8kQwfMXZ0Ho1DO4 model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1DJFb1AW3hrUNrXW4zm9ns__zVB8D91-A
From (redirected): https://drive.google.com/uc?id=1DJFb1AW3hrUNrXW4zm9ns__zVB8D91-A&confirm=t&uuid=6777b45d-bd17-4e28-b3bc-11852730b8c5
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_timeattention/HSN_eca_nfnet_l1_2026-03-09_161144/models/model.pth
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 192M/192M [00:03<00:00, 63.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1cjhVqNc4oVYkZNsgnliVnQcQ-D4pKNYF
From (redirected): https://drive.google.com/uc?id=1cjhVqNc4oVYkZNsgnliVnQcQ-D4pKNYF&confirm=t&uuid=4fbdf99e-6b94-453d-915a-8ebe528a3a7b
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_timeattention/HSN_eca_nfnet_l1_2026-03-09_162312/models/model.pth
100%|█████████████████████████████████████████

['../ckpts/rest/HSN_timeattention/HSN_eca_nfnet_l1_2026-03-09_162312', '../ckpts/rest/HSN_timeattention/HSN_eca_nfnet_l1_2026-03-09_161144', '../ckpts/rest/HSN_timeattention/HSN_eca_nfnet_l1_2026-03-09_163440']


2026-05-05 10:43:58,886 | INFO | Task: HSN | Number of events: 12000
Testing: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 94/94 [00:43<00:00,  2.17it/s]


{'HSN': {'auroc': 0.9457022326757826, 'cmap': 0.5847186219686932, 'top1_acc': 0.7046060164769491}}


In [12]:
# SSA
if not os.path.exists('../ckpts/rest/HSN_SSA'):
    gdown.download_folder('https://drive.google.com/drive/folders/1uxiSKvuclFVgqn1_oVvZZixUfKRnBE0f?usp=sharing', output="../ckpts/rest/")

ckpts = glob.glob('../ckpts/rest/HSN_SSA/*')


run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 11qRm0eBHT3eVoIWFvxkdY9XaSAFBfsjI HSN_eca_nfnet_l1_2026-03-14_102758
Retrieving folder 1wBmjM-eFYlHeOyxFC1Lel9BDoGh6CbXP models
Processing file 1TwKaMBBmwgr51Q2Vx03NZznu9YgF_oJR model.pth
Retrieving folder 1vvX9O91-_Hmrnvm-PWLR0Lu0LuL-e5TS HSN_eca_nfnet_l1_2026-03-14_103929
Retrieving folder 1oIsbBqd7EhRxKGA_9Ae6bW8vAK7IBM0W models
Processing file 1LxrSgSAbeOn1gmNI3LLoQByHAb0cOaR0 model.pth
Retrieving folder 1taxllrw359iaGWFXSrQ3orUQWy83lFUk HSN_eca_nfnet_l1_2026-03-14_105101
Retrieving folder 1nCgIH54CGF92_GYr8NG9Ra9xQKZEv8Q2 models
Processing file 1m3N8YEnBrc5TlS0gEi5JQkMx_cPpgeV- model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1TwKaMBBmwgr51Q2Vx03NZznu9YgF_oJR
From (redirected): https://drive.google.com/uc?id=1TwKaMBBmwgr51Q2Vx03NZznu9YgF_oJR&confirm=t&uuid=5758d94b-9be2-40df-81d0-ebc66bc20ead
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_SSA/HSN_eca_nfnet_l1_2026-03-14_102758/models/model.pth
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 192M/192M [00:02<00:00, 82.8MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LxrSgSAbeOn1gmNI3LLoQByHAb0cOaR0
From (redirected): https://drive.google.com/uc?id=1LxrSgSAbeOn1gmNI3LLoQByHAb0cOaR0&confirm=t&uuid=5d2703df-2d26-441f-9857-0c8f8fd6c92a
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_SSA/HSN_eca_nfnet_l1_2026-03-14_103929/models/model.pth
100%|█████████████████████████████████████████████████████████████

['../ckpts/rest/HSN_SSA/HSN_eca_nfnet_l1_2026-03-14_105101', '../ckpts/rest/HSN_SSA/HSN_eca_nfnet_l1_2026-03-14_102758', '../ckpts/rest/HSN_SSA/HSN_eca_nfnet_l1_2026-03-14_103929']


2026-05-05 10:46:42,775 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:46:42,895 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:46:42,907 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:46:43,215 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:46:43,334 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:46:43,345 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:46:43,662 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:46:43,774 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9551479085393016, 'cmap': 0.5842041371826746, 'top1_acc': 0.724009652932485}}


## 3. Effect of temporal context length during testing, showing how segment extension enhances model performance (Table 6)

In [13]:
# DFA using a 7-second input, where the centered 5 seconds are the target samples.
# target_length is the time resolution of the spectrograms
ckpts = glob.glob('../ckpts/DT/HSN/*')
run_testing(ckpts, extracted_interval=7, target_length=701, device=DEVICE)

['../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-05-05 10:49:01,935 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:49:02,056 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:49:02,061 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:49:02,407 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:49:02,525 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:49:02,539 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:49:02,907 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:49:03,027 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9560262623085944, 'cmap': 0.5792627376801449, 'top1_acc': 0.7324406305948893}}


In [14]:
# DFA using the original 5-second input of the test samples.
run_testing(ckpts, extracted_interval=5, target_length=501, device=DEVICE)

['../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-05-05 10:51:21,504 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:51:21,628 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:51:21,643 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:51:22,005 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:51:22,128 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:51:22,142 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:51:22,505 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:51:22,624 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9524781767262779, 'cmap': 0.5791612980916198, 'top1_acc': 0.7189263105392456}}


## 4. Ablation study on data augmentation methods, evaluating their impact on bird species recognition using HSN test data (Table 7)

In [15]:
# Baseline: No Augmentation

if not os.path.exists('../ckpts/rest/HSN_noaug'):
    gdown.download_folder('https://drive.google.com/drive/folders/1o6F1FxLb4vrBle1TU-U41x42QQx756ge?usp=sharing', output="../ckpts/rest/")


ckpts = glob.glob('../ckpts/rest/HSN_noaug/*')
run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 1BjcMSJe44xUeWYhaFRGoCTtvvi1qxmef HSN_eca_nfnet_l1_2025-10-27_083456
Retrieving folder 1ALf5XaIAjYBBVX7cS8X7XPwpp-_qyvL7 models
Processing file 1HiS6IrSx3Ig_MW_qNnXr0Cc4PL4kqoTJ model.pth
Retrieving folder 1RSn5eTggp6gs3uDv3withBUYwELAildS HSN_eca_nfnet_l1_2025-10-27_084621
Retrieving folder 1gZ74_pWus1NlCnJ8LyfhZDwaTeq230KD models
Processing file 1w4oe6TlHcDs4eeeYU_LokccHJWn3ZX9e model.pth
Retrieving folder 1uBkclQH7Xh8wpGhqMaanTvPgiVW_kOPq HSN_eca_nfnet_l1_2025-10-27_085747
Retrieving folder 1XDOAKU_sWEUDAM9EJVFgigkqF_VN1rUK models
Processing file 1ogjt6kQLIdiaITrNgHoUWHL5X6ULmybs model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1HiS6IrSx3Ig_MW_qNnXr0Cc4PL4kqoTJ
From (redirected): https://drive.google.com/uc?id=1HiS6IrSx3Ig_MW_qNnXr0Cc4PL4kqoTJ&confirm=t&uuid=04a3ca3f-92c6-4b35-8b22-33298bb1b6bb
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_noaug/HSN_eca_nfnet_l1_2025-10-27_083456/models/model.pth
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 230M/230M [00:05<00:00, 40.8MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1w4oe6TlHcDs4eeeYU_LokccHJWn3ZX9e
From (redirected): https://drive.google.com/uc?id=1w4oe6TlHcDs4eeeYU_LokccHJWn3ZX9e&confirm=t&uuid=299b4664-d4f6-4d22-adc6-78ff36ce2c8f
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_noaug/HSN_eca_nfnet_l1_2025-10-27_084621/models/model.pth
100%|█████████████████████████████████████████████████████████

['../ckpts/rest/HSN_noaug/HSN_eca_nfnet_l1_2025-10-27_084621', '../ckpts/rest/HSN_noaug/HSN_eca_nfnet_l1_2025-10-27_083456', '../ckpts/rest/HSN_noaug/HSN_eca_nfnet_l1_2025-10-27_085747']


2026-05-05 10:54:20,332 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:54:20,462 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:54:20,476 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:54:20,827 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:54:20,942 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:54:20,956 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:54:21,311 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:54:21,430 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.8588290967953124, 'cmap': 0.5249688989157181, 'top1_acc': 0.6417457063992819}}


In [16]:
# +Mixup (Signal)

if not os.path.exists('../ckpts/rest/HSN_sigmixup'):
    gdown.download_folder('https://drive.google.com/drive/folders/1hBOIo1XBc94B1penreZcFqWsMwsVlLJ1?usp=sharing', output="../ckpts/rest/")


ckpts = glob.glob('../ckpts/rest/HSN_sigmixup/*')
run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 1bCIZqZeKTfaJkUYv9Y7d9ivOv3vvKTqj HSN_eca_nfnet_l1_2025-10-27_095651
Retrieving folder 1ByPZfP28Hu6rnFr-RHQDMTC5WMc4Qmc9 models
Processing file 1GCIA86oBNVOBIHvUpkfECBxeScX60CBV model.pth
Retrieving folder 1nVpiOBCdKYEpMrexb98YDk0o2b8x5oOg HSN_eca_nfnet_l1_2025-10-27_100822
Retrieving folder 1qInDZCh3EONgDsZDtk8Z9xE7EnRXdfwo models
Processing file 11AHuOz1Zs-ZiXFYXbgwjlZZTm_jPFzcN model.pth
Retrieving folder 1tDptWga3WbQe4gon4q0rZ3zKSHkAbt_0 HSN_eca_nfnet_l1_2025-10-27_101953
Retrieving folder 1KEi2ejtC0Ja5ETJfEViOhSC3q9-UeHJZ models
Processing file 1INdHVJbHBqOx7JU5FIaLxvQ1Fybh3n-j model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1GCIA86oBNVOBIHvUpkfECBxeScX60CBV
From (redirected): https://drive.google.com/uc?id=1GCIA86oBNVOBIHvUpkfECBxeScX60CBV&confirm=t&uuid=f58fd8bd-6211-4d0d-afa1-53f95dc26741
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_sigmixup/HSN_eca_nfnet_l1_2025-10-27_095651/models/model.pth
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 230M/230M [00:02<00:00, 91.9MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=11AHuOz1Zs-ZiXFYXbgwjlZZTm_jPFzcN
From (redirected): https://drive.google.com/uc?id=11AHuOz1Zs-ZiXFYXbgwjlZZTm_jPFzcN&confirm=t&uuid=5e498aff-3423-4e9f-9dc3-09f99e58ca5b
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_sigmixup/HSN_eca_nfnet_l1_2025-10-27_100822/models/model.pth
100%|███████████████████████████████████████████████████

['../ckpts/rest/HSN_sigmixup/HSN_eca_nfnet_l1_2025-10-27_100822', '../ckpts/rest/HSN_sigmixup/HSN_eca_nfnet_l1_2025-10-27_095651', '../ckpts/rest/HSN_sigmixup/HSN_eca_nfnet_l1_2025-10-27_101953']


2026-05-05 10:57:15,038 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:57:15,159 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:57:15,172 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:57:15,556 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:57:15,675 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 10:57:15,686 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 10:57:16,061 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 10:57:16,175 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9059694394473373, 'cmap': 0.5447450570350201, 'top1_acc': 0.7146488030751547}}


In [17]:
# +Mixup (Signal) +Background-noise

if not os.path.exists('../ckpts/rest/HSN_sigmixup+bgnoise'):
    gdown.download_folder('https://drive.google.com/drive/folders/12mt86SVO8yDIpRSjhSNLzrqq_JT4EaAN?usp=sharing', output="../ckpts/rest/")


ckpts = glob.glob('../ckpts/rest/HSN_sigmixup+bgnoise/*')
run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 1jJhEgrYBKhfw1B3RPzmLbewDbC2yVogE HSN_eca_nfnet_l1_2025-10-27_103302
Retrieving folder 1JF4vFfdLU4cZgQl1GBsy29cez8at0axm models
Processing file 1JYpkOQg-BTgzwErRuKH_EWZNUS7yxEMy model.pth
Retrieving folder 1ZzNpvpnptdOxksmHuYgul6z0Ev9COxW7 HSN_eca_nfnet_l1_2025-10-27_104438
Retrieving folder 1gkXWRsYBa-S3hbOzb7hhcyDG1joiQ-s5 models
Processing file 18xmdlWfopVyHTfkoKGuroEcKGc1fuIUp model.pth
Retrieving folder 1x1J-DWKWlWJmMn3e6JNyYvVkVSVnS6bY HSN_eca_nfnet_l1_2025-10-27_105614
Retrieving folder 1JkEopjXL1lkj7xbMT9by9CcB8fMsWMg6 models
Processing file 15q1xCiRxLg3N8f2wwKxpaR5wg_DkYFih model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1JYpkOQg-BTgzwErRuKH_EWZNUS7yxEMy
From (redirected): https://drive.google.com/uc?id=1JYpkOQg-BTgzwErRuKH_EWZNUS7yxEMy&confirm=t&uuid=8be54660-0cda-4511-bc47-1cc287a28afd
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_sigmixup+bgnoise/HSN_eca_nfnet_l1_2025-10-27_103302/models/model.pth
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 230M/230M [00:02<00:00, 89.8MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=18xmdlWfopVyHTfkoKGuroEcKGc1fuIUp
From (redirected): https://drive.google.com/uc?id=18xmdlWfopVyHTfkoKGuroEcKGc1fuIUp&confirm=t&uuid=11ef3356-cbc3-4d5c-aa7c-70bf8b41c1ff
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_sigmixup+bgnoise/HSN_eca_nfnet_l1_2025-10-27_104438/models/model.pth
100%|███████████████████████████████████

['../ckpts/rest/HSN_sigmixup+bgnoise/HSN_eca_nfnet_l1_2025-10-27_103302', '../ckpts/rest/HSN_sigmixup+bgnoise/HSN_eca_nfnet_l1_2025-10-27_105614', '../ckpts/rest/HSN_sigmixup+bgnoise/HSN_eca_nfnet_l1_2025-10-27_104438']


2026-05-05 11:00:14,003 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 11:00:14,130 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 11:00:14,145 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 11:00:14,522 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 11:00:14,640 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 11:00:14,655 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 11:00:15,038 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 11:00:15,157 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9062293678286514, 'cmap': 0.5517955212866806, 'top1_acc': 0.7202281554539999}}


In [18]:
# +Mixup (Signal) +Background noise +colored noise +gain

if not os.path.exists('../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain'):
    gdown.download_folder('https://drive.google.com/drive/folders/13Sy9zOTHx1PPkhesRaQnb-gSYQMvbE0L?usp=sharing', output="../ckpts/rest/")


ckpts = glob.glob('../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain/*')
run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 11iZiOOIIdO4wZuEO67eBHHVu77MM11Uu HSN_eca_nfnet_l1_2026-03-09_203948
Retrieving folder 1Vw5J30OTbOesjmAqj-rE1Q-iFj8LqNU1 models
Processing file 1l51D7oo62rZ4pMGe06Uog3U1gYYtOMjy model.pth
Retrieving folder 1YzLguLjIzIyiWnvqc9WDTwk3iNu_aZUX HSN_eca_nfnet_l1_2026-03-09_205253
Retrieving folder 1fUuepXBeP-mGafAmRFIvLTD3RVD5drjQ models
Processing file 1NnrO3ka5yG8oUZYBZzk6WsVAFlOR8lVk model.pth
Retrieving folder 1uGPBjIJ_tI3HfuEgP1rZQxd5XEk8ogNd HSN_eca_nfnet_l1_2026-03-09_210951
Retrieving folder 1O7VqI_3hOCRnZ4Ws315itRmhbHOL8cZP models
Processing file 1Wqb2_rBb9FwQ-DZMbVWgKrVleCbr9OZ8 model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1l51D7oo62rZ4pMGe06Uog3U1gYYtOMjy
From (redirected): https://drive.google.com/uc?id=1l51D7oo62rZ4pMGe06Uog3U1gYYtOMjy&confirm=t&uuid=eb8882e3-b807-453c-9a23-ebfbc8809ed2
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain/HSN_eca_nfnet_l1_2026-03-09_203948/models/model.pth
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 230M/230M [00:06<00:00, 35.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1NnrO3ka5yG8oUZYBZzk6WsVAFlOR8lVk
From (redirected): https://drive.google.com/uc?id=1NnrO3ka5yG8oUZYBZzk6WsVAFlOR8lVk&confirm=t&uuid=c77c5b6e-5f7f-4226-bd18-927f9ab4cbf5
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain/HSN_eca_nfnet_l1_2026-03-09_205253/models/model.pth
100%

['../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain/HSN_eca_nfnet_l1_2026-03-09_205253', '../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain/HSN_eca_nfnet_l1_2026-03-09_203948', '../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain/HSN_eca_nfnet_l1_2026-03-09_210951']


2026-05-05 11:02:54,434 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 11:02:54,552 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 11:02:54,562 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 11:02:54,927 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 11:02:55,040 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 11:02:55,053 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 11:02:55,417 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 11:02:55,536 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9115621064078656, 'cmap': 0.5518943480274384, 'top1_acc': 0.7261794209480286}}


In [19]:
# +Mixup (Signal) +Background-noise +colored-noise +gain +no-call 

if not os.path.exists('../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall'):
    gdown.download_folder('https://drive.google.com/drive/folders/1I7Bpd1hDMxyoaqdGyM5Nqn0onxUnZtce?usp=sharing', output="../ckpts/rest/")


ckpts = glob.glob('../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/*')
run_testing(ckpts, device=DEVICE)

Retrieving folder contents


Retrieving folder 1JLALb5hEeOT14auoTNsX_eCXpC0UocZU HSN_eca_nfnet_l1_2026-03-10_113228
Retrieving folder 1q3tM7NSAuy2I93rngKsmCUPr5ySmG5Ne models
Processing file 11kDnBttFu8IP1o22_sjhLh24z3KDIe1g model.pth
Retrieving folder 1RLhOcIAbOlCmok_DeOG13ETOe97O-Aej HSN_eca_nfnet_l1_2026-03-10_114410
Retrieving folder 1hJvpOiddyEirt7UIc4dPFXrUPL8raGpO models
Processing file 1KRF8Ohh9OpP0-Uef8Vwrdb1kgVBisATT model.pth
Retrieving folder 1KWyeevdnAM3yWbvLh0XOypjuO7rigQUW HSN_eca_nfnet_l1_2026-03-10_115551
Retrieving folder 1oNN6ivP9yVpwtKF1PIt_LFRieERtYhY6 models
Processing file 1IdIPT7x0sDmeXthN_n_TQs00yRr4OsWS model.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=11kDnBttFu8IP1o22_sjhLh24z3KDIe1g
From (redirected): https://drive.google.com/uc?id=11kDnBttFu8IP1o22_sjhLh24z3KDIe1g&confirm=t&uuid=f0be7b9d-f2fa-4a5a-8165-70596bb10af5
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/HSN_eca_nfnet_l1_2026-03-10_113228/models/model.pth
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 230M/230M [00:10<00:00, 22.5MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1KRF8Ohh9OpP0-Uef8Vwrdb1kgVBisATT
From (redirected): https://drive.google.com/uc?id=1KRF8Ohh9OpP0-Uef8Vwrdb1kgVBisATT&confirm=t&uuid=7b7af96a-113d-4845-9329-d6cd4fc65340
To: /home/bellafkir/Documents/sa4birds/ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/HSN_eca_nfnet_l1_2026-03-10_114410/models/

['../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/HSN_eca_nfnet_l1_2026-03-10_113228', '../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/HSN_eca_nfnet_l1_2026-03-10_115551', '../ckpts/rest/HSN_sigmixup+bgnoise+colorednoise+gain+nocall/HSN_eca_nfnet_l1_2026-03-10_114410']


2026-05-05 11:05:42,313 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 11:05:42,434 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 11:05:42,446 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 11:05:42,794 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 11:05:42,936 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 11:05:42,950 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 11:05:43,301 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 11:05:43,422 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9506547736325318, 'cmap': 0.5801166357128348, 'top1_acc': 0.7243196368217468}}


In [20]:
# +Mixup (Signal) +Background-noise +colored-noise +gain +no-call +Mixup (Spec) +SpecAug

ckpts = glob.glob('../ckpts/DT/HSN/*')
run_testing(ckpts, device=DEVICE)

['../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_113316', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_112131', '../ckpts/DT/HSN/HSN_eca_nfnet_l1_2025-10-20_114501']


2026-05-05 11:07:59,453 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 11:07:59,576 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 11:07:59,591 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 11:07:59,945 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 11:08:00,063 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
2026-05-05 11:08:00,077 | INFO | Converted input conv stem.conv1 pretrained weights from 3 to 1 channel(s)
2026-05-05 11:08:00,439 | INFO | Loading pretrained weights from Hugging Face hub (timm/eca_nfnet_l1.ra2_in1k)
2026-05-05 11:08:00,564 | INFO | [timm/eca_nfnet_l1.ra2_in1k] Safe alternative available for 'pytorch_mod

{'HSN': {'auroc': 0.9560262623085944, 'cmap': 0.5792627376801449, 'top1_acc': 0.7324406305948893}}
